# Complete PGM End-to-End Pipeline (Single Notebook)

**Merged notebook** — everything in one place:

| Module | Topic | Dataset |
|--------|-------|---------|
| **Module 1** | Static Bayesian Network (course reference) | Breast Cancer Wisconsin |
| **Module 2** | Dynamic Bayesian Network (your project) | COVID-19 (Corona) contact tracing |

**Libraries:** pgmpy, scikit-learn, networkx, numpy, pandas, matplotlib

---

### Project goal (Module 2)

> Model disease spread using a DBN with latent **SEIR** states and **COVID test** observations. Answer: *P(node i is infectious | all test observations)?*

---

Run all cells **top to bottom**.


In [ ]:
pip install pgmpy networkx matplotlib numpy pandas scipy scikit-learn


LECTURE OUTLINE (complete pipeline)

**MODULE 1 — Static BN (reference pattern)**
* PART 1  DATA & PREPROCESSING   – Breast Cancer Wisconsin, discretisation
* PART 2  REPRESENTATION          – Bayesian Network theory
* PART 3  STRUCTURE LEARNING      – Hill-Climb + BIC
* PART 4  PARAMETER LEARNING      – Maximum Likelihood Estimation (MLE)
* PART 5  INFERENCE               – Variable Elimination
* PART 6  EVALUATION              – classification accuracy

**MODULE 2 — Dynamic BN (COVID-19 / Corona project)**
* PART 1  DATA & PREPROCESSING   – Corona contact tracing + OWID context
* PART 2  REPRESENTATION          – 2-time-slice DBN, SEIR CPTs
* PART 3  MODEL STRUCTURE         – contact network from Corona data
* PART 4  PARAMETER LEARNING      – EM algorithm (beta, sigma, gamma)
* PART 5  INFERENCE               – Variable Elimination + forward-backward
* PART 6  EVALUATION & FIGURES    – epidemic curves, heatmaps, dashboard


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import textwrap
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "src").is_dir() else (_cwd.parent if (_cwd.parent / "src").is_dir() else _cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import networkx as nx

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# scikit-learn (Module 1)
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# pgmpy (both modules)
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.estimators import HillClimbSearch
from pgmpy.parameter_estimator import DiscreteMLE
from pgmpy.inference import VariableElimination
from pgmpy.models import DynamicBayesianNetwork as DBN

# Project src (Module 2)
from src.config import ModelParams, SimConfig, STATES, STATE_IDX
from src.config import OBS_MISSING, OBS_POS, OBS_NEG
from src.corona_data import (
    download_corona_dataset, load_corona_dataset,
    load_corona_contact_tables, load_corona_owid_context, CORONA_DIR,
)
from src.model import build_dbn_structure, transition_distribution, emission_likelihood, export_pgmpy_dbn
from src.network import network_summary
from src.inference import infer_infectious_probability, query_node_infectious
from src.learning import em_learn
from src.simulation import epidemic_counts

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Output dir  :", OUTPUT_DIR)


---

# MODULE 1 — Static Bayesian Network (Course Reference)

  Dataset  : sklearn Breast Cancer Wisconsin (569 samples, 30 features)

  Model    : Discrete Bayesian Network

  Goal     : Learn DAG structure, fit CPDs, run Variable Elimination inference

---


## MODULE 1 — PART 1 – DATA & PREPROCESSING


 **1-A  Load the Wisconsin Breast Cancer dataset from sklearn**

* 569 patients, 30 continuous cell-nucleus measurements

* Target: 0 = Malignant, 1 = Benign


In [ ]:
raw    = load_breast_cancer()
df_raw = pd.DataFrame(raw.data, columns=raw.feature_names)
df_raw["diagnosis"] = raw.target          # 0 = Malignant, 1 = Benign

print(f"Dataset shape  : {df_raw.shape}")
print(f"Class balance  : {dict(zip(['Malignant','Benign'], np.bincount(raw.target)))}")
print(f"\nFirst 3 rows (continuous):\n{df_raw.head(3).to_string()}\n")




**M1-1-B  Feature selection**
  *  We keep 6 features that have high discriminatory power for a
 * readable graph. Students can extend this to all 30.


In [ ]:

FEATURES = [
    "mean radius",
    "mean texture",
    "mean concavity",
    "mean area",
    "worst radius",
    "worst concave points",
    ]
TARGET = "diagnosis"





**M1-1-C  Discretisation**
* Bayesian Networks with discrete CPDs require categorical states.
* We bin each continuous feature into 3 quantile-equal buckets:
* "0.0" = Low   "1.0" = Medium   "2.0" = High


In [ ]:

kbd  = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
disc = kbd.fit_transform(df_raw[FEATURES])

df = pd.DataFrame(disc.astype(float).astype(str), columns=FEATURES)
df[TARGET] = df_raw[TARGET].astype(str).values   # "0" = Malignant, "1" = Benign

print("Discretised dataset (first 3 rows):")
print(df.head(3).to_string())
print(f"\nState space per variable:")
for col in df.columns:
    print(f"  {col:35s}: {sorted(df[col].unique())}")

# 1-D  Train / Test split (80 / 20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42,
                                     stratify=df[TARGET])
print(f"\nTrain size : {len(train_df)}   Test size : {len(test_df)}")


## MODULE 1 — PART 2:  REPRESENTATION  (Bayesian Network)


In [ ]:
print(textwrap.dedent("""
A Bayesian Network (BN) is a Directed Acyclic Graph (DAG) where:
  • Nodes  = random variables  (features + diagnosis)
  • Edges  = direct probabilistic dependencies  A → B means A "influences" B
  • CPDs   = Conditional Probability Distributions P(node | parents)

The joint distribution factorises as:
  P(X₁, X₂, …, Xₙ) = ∏ P(Xᵢ | Pa(Xᵢ))

This compact representation encodes conditional independences via the
Markov condition and d-separation, enabling efficient inference even in
very large models.

Key components we will build step-by-step:
  1. Structure  (the DAG)
  2. Parameters (the CPD tables)
  3. Inference  (answering probabilistic queries)
"""))



## MODULE 1 — PART 3: STRUCTURE LEARNING – Hill-Climb + BIC


In [ ]:
print(textwrap.dedent("""
Structure learning discovers which edges to add/remove/reverse to best
explain the data.

Algorithm : Greedy Hill-Climbing Search
  • Start from an empty (or random) DAG
  • At each step try: add / remove / reverse every possible edge
  • Accept the change that maximises the scoring function
  • Stop when no improvement can be found

Scoring function : BIC (Bayesian Information Criterion)
  BIC(G, D) = log-likelihood(G, D) − (k/2) · log(n)

  The penalty term (k/2)·log(n) discourages overly complex graphs,
  acting as Occam's razor.  Higher BIC = better model.
"""))

hc             = HillClimbSearch(train_df)
learned_struct = hc.estimate(scoring_method="bic-d", max_indegree=3)
learned_edges  = list(learned_struct.edges())

print("Learned edges (DAG):")
for edge in learned_edges:
    print(f"  {edge[0]}  →  {edge[1]}")


## MODULE 1 — PART 4 – PARAMETER LEARNING  (Maximum Likelihood Estimation)


In [ ]:
print(textwrap.dedent("""
Given the DAG structure, we estimate the CPD of each node from data.

Maximum Likelihood Estimation (MLE):
  P(Xᵢ = xᵢ | Pa(Xᵢ) = pa) = count(Xᵢ=xᵢ, Pa(Xᵢ)=pa) / count(Pa(Xᵢ)=pa)

This is simply a frequency count – the probability that a node takes a
particular value given its parents' values, estimated from the training data.
"""))

model = DiscreteBayesianNetwork(learned_edges)
model.fit(train_df, estimator=DiscreteMLE())

print(f"Number of nodes : {len(model.nodes())}")
print(f"Number of CPDs  : {len(model.cpds)}")
print("\nAll learned CPDs:\n" + "─" * 50)
for cpd in model.cpds:
    print(cpd)
    print()



## MODULE 1 — PART 5 – INFERENCE  (Variable Elimination)


In [ ]:
print(textwrap.dedent("""
Inference = answering probabilistic queries using the fitted model.

Variable Elimination (VE):
  • Marginal query  : P(Y | evidence)  – posterior distribution over Y
  • MAP query       : argmax_Y P(Y | evidence)  – most likely state of Y

VE works by summing out (marginalising) hidden variables one by one
using the factor representation of the CPDs, exploiting conditional
independence to avoid redundant computation.
"""))

infer = VariableElimination(model)




**M1-5-A  Marginal query: prior probability of diagnosis (no evidence)**


In [ ]:

if TARGET in model.nodes():
    q_prior = infer.query([TARGET])
    print("Prior P(diagnosis) [no evidence]:")
    print(q_prior, "\n")





**M1-5-B  Posterior given clinical observations**


In [ ]:

evidence_1 = {}
if "mean radius"    in model.nodes(): evidence_1["mean radius"]    = "2.0"  # HIGH
if "mean concavity" in model.nodes(): evidence_1["mean concavity"] = "2.0"  # HIGH
if "mean area"      in model.nodes(): evidence_1["mean area"]      = "2.0"  # HIGH

if TARGET in model.nodes() and evidence_1:
    # remove TARGET from evidence if it snuck in
    evidence_1.pop(TARGET, None)
    q1 = infer.query([TARGET], evidence=evidence_1)
    print("Posterior P(diagnosis | high radius, high concavity, high area):")
    print(q1)
    print("  → 0 = Malignant, 1 = Benign\n")

evidence_2 = {}
if "mean radius"    in model.nodes(): evidence_2["mean radius"]    = "0.0"  # LOW
if "mean concavity" in model.nodes(): evidence_2["mean concavity"] = "0.0"  # LOW

if TARGET in model.nodes() and evidence_2:
    evidence_2.pop(TARGET, None)
    q2 = infer.query([TARGET], evidence=evidence_2)
    print("Posterior P(diagnosis | low radius, low concavity):")
    print(q2)
    print("  → 0 = Malignant, 1 = Benign\n")



**M1-5-C  MAP query**


In [ ]:

if TARGET in model.nodes() and evidence_1:
    map_result = infer.map_query([TARGET], evidence=evidence_1)
    label = "Malignant" if map_result[TARGET] == "0" else "Benign"
    print(f"MAP prediction for high-risk evidence: {label} (state={map_result[TARGET]})")


## MODULE 1 — PART 6 – EVALUATION


In [ ]:
def predict(row: pd.Series) -> str:
    """Return MAP prediction for a single test row."""
    evidence = {
        feat: row[feat]
        for feat in FEATURES
        if feat in model.nodes() and feat != TARGET
    }
    try:
        result = infer.map_query([TARGET], evidence=evidence)
        return result[TARGET]
    except Exception:
        return "1"   # fallback: predict Benign if inference fails


y_true = test_df[TARGET].values
y_pred = [predict(row) for _, row in test_df.iterrows()]

acc = accuracy_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred, labels=["0", "1"])

print(f"Accuracy : {acc:.4f}  ({acc*100:.1f}%)\n")
print("Confusion Matrix (rows = True, cols = Predicted):")
print(f"           Pred-Malignant  Pred-Benign")
print(f"  True-Malignant  {cm[0,0]:5d}          {cm[0,1]:5d}")
print(f"  True-Benign     {cm[1,0]:5d}          {cm[1,1]:5d}\n")
print("Full Classification Report:")
print(classification_report(y_true, y_pred,
                             target_names=["Malignant(0)", "Benign(1)"]))


---

# MODULE 2 — Dynamic Bayesian Network (COVID-19 / Corona Project)

  Dataset  : COVID-19 Geneva contact tracing + OWID Switzerland context

  Model    : 2-time-slice SEIR Dynamic Bayesian Network on a contact network

  Goal     : Representation + Inference + EM learning on real Corona epidemic data

---


## MODULE 2 — PART 1 – DATA & PREPROCESSING


**M2-1-A  Load the COVID-19 (Corona) dataset**

* Contact-tracing CSVs: individuals, close contacts, positive PCR test dates
* OWID Switzerland: national COVID-19 case curve for context


In [ ]:
print(textwrap.dedent("""
MODULE 2 — COVID-19 (CORONA) DATASET
  Latent X_i^t  = SEIR state (Susceptible, Exposed, Infectious, Recovered)
  Observed Y_i^t = COVID test (positive / negative / missing)
  Network        = documented close contacts from tracing
"""))

download_corona_dataset()
suivi, entourage = load_corona_contact_tables()
print(f"Tracing records : {len(suivi):,}  |  Contact rows : {len(entourage):,}")
print(f"Data folder     : {CORONA_DIR}")
print(suivi[["record_id_pos","date_res","contact_record_id"]].dropna(subset=["date_res"]).head(3))

owid = load_corona_owid_context("Switzerland")
owid_sub = owid[(owid["date"]>="2020-02-01")&(owid["date"]<="2020-06-30")]
fig, ax = plt.subplots(figsize=(9,3))
ax.plot(owid_sub["date"], owid_sub["new_cases"], color="#c0392b", lw=1.5)
ax.set_title("COVID-19 Switzerland — daily new cases (OWID)"); ax.grid(alpha=0.3)
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


**M2-1-B  Build outbreak subgraph and observation matrix Y**


In [ ]:
bundle = load_corona_dataset(max_nodes=30)
G, Y_obs, X_true = bundle.graph, bundle.Y_obs, bundle.X_true
patient_zero, meta = bundle.patient_zero, bundle.metadata
params = ModelParams(beta=0.30, sigma=0.20, gamma=0.10)

print(f"Dataset      : {bundle.dataset_name}")
print(f"Nodes / days : {meta['n_nodes']} people, {meta['n_timesteps']} timesteps")
print(f"Date range   : {meta['date_start']} -> {meta['date_end']}")
print(f"Pos. tests   : {meta['n_positive_tests']}  |  Matrix {Y_obs.shape}")
print(f"Patient zero : node {patient_zero}")


**M2-1-C  List positive COVID test observations**


In [ ]:
obs_rows = []
for t in range(Y_obs.shape[0]):
    for i in range(Y_obs.shape[1]):
        if Y_obs[t,i] == OBS_POS:
            obs_rows.append({"node":i, "person_id":bundle.node_ids[i],
                "day_t":t, "date":str(bundle.dates[t].date()), "test":"POSITIVE"})
display(pd.DataFrame(obs_rows))
counts = epidemic_counts(X_true)
print(f"Peak infectious (approx): {max(counts['I'])}")


## MODULE 2 — PART 2 – REPRESENTATION  (Dynamic Bayesian Network)


In [ ]:
print(textwrap.dedent("""
A Dynamic Bayesian Network (DBN) unrolls a 2-time-slice template:

  X_i^{t-1} -> X_i^t       within-person SEIR dynamics
  X_j^{t-1} -> X_i^t       transmission (contact network)
  X_i^t     -> Y_i^t       COVID test emission

SEIR latent states: S, E, I, R
Parameters: beta (transmission), sigma (E->I), gamma (I->R)
"""))


## MODULE 2 — PART 3 – MODEL STRUCTURE


In [ ]:
structure = build_dbn_structure(G)
summary = network_summary(G)
print(structure["description"])
print(summary)
dbn = export_pgmpy_dbn(G, params)
print(f"pgmpy DBN: {len(list(dbn.nodes()))} node variables, {len(list(dbn.edges()))} edges")

pos = nx.spring_layout(G, seed=42)
fig, ax = plt.subplots(figsize=(8,6))
nx.draw(G, pos, with_labels=True, node_color="#a8d4f0", node_size=550, font_size=9, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=[patient_zero], node_color="#e74c3c", node_size=650, ax=ax)
ax.set_title("FIGURE M2-0 — COVID-19 Contact Network (red = patient zero)")
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"fig0_network.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(9,4.5))
ax.plot(counts["I"], label="Infectious", lw=2.5, color="#c0392b")
ax.plot(counts["E"], label="Exposed", ls="--", color="#e67e22")
ax.set_title("FIGURE M2-1 — Epidemic Curve"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"fig1_epidemic_curve.png", dpi=150); plt.show()


## MODULE 2 — PART 4 – PARAMETER LEARNING  (EM Algorithm)


In [ ]:
print(textwrap.dedent("""
Module 1 used MLE on observed features.  Module 2 uses EM because
SEIR states are LATENT — we only see COVID test results.

E-step: forward-backward inference -> soft state counts
M-step: re-estimate beta, sigma, gamma
"""))
init_p = ModelParams(0.10, 0.10, 0.10)
learned, history = em_learn(G, Y_obs, init_p, n_iter=25, patient_zero=patient_zero, verbose=True)
print(f"Learned: beta={learned.beta:.3f} sigma={learned.sigma:.3f} gamma={learned.gamma:.3f}")

fig, ax = plt.subplots(figsize=(9,4.5))
for idx, (lab,col) in enumerate(zip(["beta","sigma","gamma"],["#2980b9","#8e44ad","#16a085"])):
    ax.plot(history[:,idx], label=lab, color=col, lw=2)
ax.set_title("FIGURE M2-4 — EM Convergence"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"fig4_em_convergence.png", dpi=150); plt.show()


## MODULE 2 — PART 5 – INFERENCE


In [ ]:
print(textwrap.dedent("""
Module 1: Variable Elimination on static BN.
Module 2: VE on a COVID test model + forward-backward on full network.

PROJECT QUERY: P(X_i^t = Infectious | all COVID test observations)
"""))

# pgmpy VE demo (parallel to Module 1 Part 5)
covid_bn = DiscreteBayesianNetwork([("Health","COVID_Test")])
covid_bn.add_cpds(
    TabularCPD("Health", 2, [[0.7],[0.3]], state_names={"Health":["NotInfectious","Infectious"]}),
    TabularCPD("COVID_Test", 2, [[0.95,0.10],[0.05,0.90]],
        evidence=["Health"], evidence_card=[2],
        state_names={"COVID_Test":["Negative","Positive"],"Health":["NotInfectious","Infectious"]}),
)
assert covid_bn.check_model()
infer_ve = VariableElimination(covid_bn)
print("P(Health | Positive test):"); print(infer_ve.query(["Health"], evidence={"COVID_Test":"Positive"}))

# Full temporal inference on Corona data
P_I, beliefs = infer_infectious_probability(G, Y_obs, params, patient_zero=patient_zero, smooth=True)
print(f"\nFull network P(I) shape: {P_I.shape}")

rows = []
for node, t in [(patient_zero,5),(patient_zero,15),(1,10),(5,20)]:
    if t >= Y_obs.shape[0]: continue
    p = query_node_infectious(G, Y_obs, params, node=node, time=t, patient_zero=patient_zero)
    rows.append({"node":node,"day":t,"P(infectious|Y)":round(p,4),
        "test":{OBS_POS:"+",OBS_NEG:"-",OBS_MISSING:"?"}[Y_obs[t,node]]})
display(pd.DataFrame(rows))


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
im = ax.imshow(P_I.T, aspect="auto", origin="lower", cmap="Reds", vmin=0, vmax=1)
ax.set_title("FIGURE M2-2 — P(Infectious | COVID Observations)")
fig.colorbar(im, ax=ax, label="P(I)")
for t in range(Y_obs.shape[0]):
    for i in range(Y_obs.shape[1]):
        if Y_obs[t,i]==OBS_POS: ax.plot(t,i,"wo",ms=4,mec="k",mew=0.5)
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"fig2_heatmap_P_I.png", dpi=150); plt.show()

max_P = P_I.max(axis=0)
fig, ax = plt.subplots(figsize=(8,6))
nodes = nx.draw_networkx_nodes(G, pos, node_color=max_P, cmap=plt.cm.Reds, vmin=0, vmax=1, node_size=600, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.35, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
fig.colorbar(nodes, ax=ax, label="max P(I)")
ax.set_title("FIGURE M2-3 — Network by Posterior P(Infectious)")
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"fig3_network_posterior.png", dpi=150); plt.show()


## MODULE 2 — PART 6 – EVALUATION & DASHBOARD


In [ ]:
fig = plt.figure(figsize=(14,10))
gs = gridspec.GridSpec(2,2, figure=fig, hspace=0.35, wspace=0.3)
ax0 = fig.add_subplot(gs[0,0])
nx.draw(G, pos, with_labels=True, node_color="#a8d4f0", node_size=350, font_size=7, ax=ax0)
nx.draw_networkx_nodes(G, pos, nodelist=[patient_zero], node_color="#e74c3c", node_size=400, ax=ax0)
ax0.set_title("A  Contact Network")
ax1 = fig.add_subplot(gs[0,1])
ax1.plot(counts["I"], color="#c0392b", lw=2, label="I"); ax1.plot(counts["E"], color="#e67e22", ls="--", label="E")
ax1.legend(fontsize=8); ax1.set_title("B  Epidemic Curve"); ax1.grid(alpha=0.3)
ax2 = fig.add_subplot(gs[1,0])
im = ax2.imshow(P_I.T, aspect="auto", origin="lower", cmap="Reds", vmin=0, vmax=1)
ax2.set_title("C  P(I) Heatmap"); fig.colorbar(im, ax=ax2, fraction=0.046)
ax3 = fig.add_subplot(gs[1,1])
for idx,(lab,col) in enumerate(zip(["beta","sigma","gamma"],["#2980b9","#8e44ad","#16a085"])):
    ax3.plot(history[:,idx], label=lab, color=col, lw=2)
ax3.set_title("D  EM Convergence"); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)
fig.suptitle("MODULE 2 — COVID-19 DBN Dashboard", fontsize=14, y=1.01)
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"fig_dashboard.png", dpi=150, bbox_inches="tight"); plt.show()


---

# FINAL SUMMARY — Complete Pipeline

| Module | PGM pillars | Method |
|--------|-------------|--------|
| **1 — Static BN** | Representation, Structure Learning, MLE, VE Inference | Breast Cancer dataset |
| **2 — Dynamic BN** | Representation, EM Learning, Belief Propagation | COVID-19 Corona dataset |

### Module 2 project query answered

> *Given COVID test observations across the contact network, what is P(node i is infectious)?*

See Module 2 Part 5 query table and Figures M2-2, M2-3.

**End of complete pipeline.** All figures saved to `outputs/notebook/`.
